# Veraxus Stage 4 — Fine-Tuning Qwen2.5-Coder (Colab L4 GPU -> Ollama GGUF)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Phongkarazz/AI-business-agent-/blob/main/training/Veraxus_Stage4_Colab_L4_Train.ipynb)

**Hướng dẫn nhanh (1-Click):**
1. Vào **Runtime → Change runtime type → Chọn `L4 GPU` (24GB VRAM)**.
2. Giữ `MODEL_PROFILE = "3b"` (tối ưu chạy mượt trên máy Mac 8GB RAM; đổi `"7b"` nếu dùng máy >=16GB RAM).
3. Bấm **Runtime → Run all** (thời gian chạy ~25–45 phút), rồi tải file `.gguf` + `Modelfile` về nạp vào Ollama.


In [ ]:
# @title 1. Cấu hình & Kiểm tra GPU L4
import torch

MODEL_PROFILE = '3b'  # @param ['3b', '7b']
RUN_DPO_STAGE = True  # @param {type:'boolean'}
SAVE_TO_GOOGLE_DRIVE = False  # @param {type:'boolean'}
DRIVE_EXPORT_DIR = '/content/drive/MyDrive/veraxus_stage4_models'  # @param {type:'string'}

REPO_RAW_BASE = 'https://raw.githubusercontent.com/Phongkarazz/AI-business-agent-/main/training/datasets'
PROFILE_MAP = {
    '3b': ('unsloth/Qwen2.5-Coder-3B-Instruct', 'veraxus-coder-3b-q4_k_m.gguf', 'veraxus-coder:3b'),
    '7b': ('unsloth/Qwen2.5-Coder-7B-Instruct', 'veraxus-coder-7b-q4_k_m.gguf', 'veraxus-coder:7b'),
}
HF_MODEL_NAME, GGUF_FILENAME, OLLAMA_TAG = PROFILE_MAP[MODEL_PROFILE]
MAX_SEQ_LENGTH = 3072
LORA_RANK = 32

assert torch.cuda.is_available(), 'Vui lòng bật GPU L4 trong Runtime -> Change runtime type!'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU: {gpu_name} ({vram_gb:.1f} GB VRAM) | bf16={torch.cuda.is_bf16_supported()}')
print(f'Target: {HF_MODEL_NAME} -> {GGUF_FILENAME} ({OLLAMA_TAG})')


In [ ]:
# @title 2. Cài đặt Unsloth + TRL (tương thích Colab L4)
!pip install -q --upgrade unsloth trl peft accelerate bitsandbytes datasets
!pip install -q --no-deps xformers


In [ ]:
# @title 3. Tải bộ dữ liệu Veraxus Stage 4 (SFT + When2Call DPO)
import gzip, json, pathlib, urllib.request

DATA_DIR = pathlib.Path('/content/veraxus_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

def load_or_fetch(filename):
    local_repo = pathlib.Path('training/datasets') / filename
    dst = DATA_DIR / filename
    if local_repo.exists():
        dst = local_repo
    elif not dst.exists():
        url = f'{REPO_RAW_BASE}/{filename}'
        print(f'Downloading {url}...')
        urllib.request.urlretrieve(url, dst)
    with gzip.open(dst, 'rt', encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]

sft_train_raw = load_or_fetch('veraxus_sft_train.jsonl.gz')
sft_eval_raw = load_or_fetch('veraxus_sft_eval.jsonl.gz')
dpo_raw = load_or_fetch('veraxus_dpo_when2call.jsonl.gz')
print(f'Loaded -> SFT train: {len(sft_train_raw)} | SFT eval: {len(sft_eval_raw)} | DPO pairs: {len(dpo_raw)}')


In [ ]:
# @title 4. Nạp Qwen2.5-Coder + QLoRA (r=32, alpha=64)
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
import torch

USE_BF16 = is_bfloat16_supported()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template='qwen-2.5')

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=2026,
)
print('LoRA model ready!')


In [ ]:
# @title 5. Stage 1 — Structured Trajectory SFT (Response-Only Loss Masking)
import inspect, re
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

_ALIAS_PAIRS = (
    ('processing_class', 'tokenizer'),
    ('eval_strategy', 'evaluation_strategy'),
    ('max_length', 'max_seq_length'),
)

def _unwrap_init(target):
    fn = target.__init__ if isinstance(target, type) else target
    seen = set()
    while fn is not None and id(fn) not in seen:
        seen.add(id(fn))
        w = getattr(fn, '__wrapped__', None)
        if w is not None:
            fn = w
            continue
        cl = getattr(fn, '__closure__', None)
        cd = getattr(fn, '__code__', None)
        if cl and cd and 'original_init' in getattr(cd, 'co_freevars', ()):
            fn = cl[cd.co_freevars.index('original_init')].cell_contents
            continue
        break
    return fn

def _safe_init(cls, raw):
    def _ep(fn):
        try:
            sig = inspect.signature(fn)
            names = {p.name for p in sig.parameters.values() if p.name != 'self' and p.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY)}
            vkw = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())
            return names, vkw
        except (TypeError, ValueError):
            return set(), True
    explicit, leaf_vkw = _ep(_unwrap_init(cls))
    if isinstance(cls, type) and (leaf_vkw or not explicit):
        for b in cls.__mro__:
            if b is not object:
                explicit.update(_ep(_unwrap_init(b))[0])
    kw = dict(raw)
    for p1, p2 in _ALIAS_PAIRS:
        if p1 in kw and p2 in kw:
            if explicit and p2 in explicit and p1 not in explicit:
                kw.pop(p1, None)
            else:
                kw.pop(p2, None)
    if explicit and (not leaf_vkw or isinstance(cls, type)):
        kw = {k: v for k, v in kw.items() if k in explicit}
    alt = {a: b for a, b in _ALIAS_PAIRS} | {b: a for a, b in _ALIAS_PAIRS}
    tried = set()
    for _ in range(8):
        try:
            return cls(**kw)
        except TypeError as exc:
            m = re.search(r"unexpected keyword argument '([^']+)'", str(exc))
            if not m or m.group(1) not in kw:
                raise
            bad = m.group(1)
            kw.pop(bad, None)
            fb = alt.get(bad)
            if fb and fb in raw and fb not in kw and fb not in tried:
                kw[fb] = raw[fb]
                tried.add(fb)
    return cls(**kw)

def fmt_sft(batch):
    return {'text': [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch['messages']]}

ds_sft_tr = Dataset.from_list([{'messages': r['messages']} for r in sft_train_raw]).map(fmt_sft, batched=True)
ds_sft_ev = Dataset.from_list([{'messages': r['messages']} for r in sft_eval_raw]).map(fmt_sft, batched=True)

sft_warmup_steps = max(5, int(0.06 * len(ds_sft_tr) * 2 / 16))
sft_args = _safe_init(SFTConfig, {
    'output_dir': '/content/veraxus_out/sft',
    'per_device_train_batch_size': 2,
    'per_device_eval_batch_size': 2,
    'gradient_accumulation_steps': 8,
    'warmup_steps': sft_warmup_steps,
    'num_train_epochs': 2,
    'learning_rate': 2e-4,
    'fp16': not USE_BF16,
    'bf16': USE_BF16,
    'logging_steps': 10,
    'eval_strategy': 'epoch',
    'evaluation_strategy': 'epoch',
    'save_strategy': 'no',
    'optim': 'adamw_8bit',
    'weight_decay': 0.01,
    'lr_scheduler_type': 'cosine',
    'seed': 2026,
    'report_to': 'none',
    'dataset_text_field': 'text',
    'max_seq_length': MAX_SEQ_LENGTH,
    'max_length': MAX_SEQ_LENGTH,
    'packing': False,
})
sft_trainer = _safe_init(SFTTrainer, {
    'model': model,
    'tokenizer': tokenizer,
    'processing_class': tokenizer,
    'train_dataset': ds_sft_tr,
    'eval_dataset': ds_sft_ev,
    'dataset_text_field': 'text',
    'max_seq_length': MAX_SEQ_LENGTH,
    'args': sft_args,
})
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)
sft_stats = sft_trainer.train()
print('SFT completed:', sft_stats.metrics)


In [ ]:
# @title 6. Stage 2 — When2Call DPO Preference Tuning (Self-Contained)
import inspect, re
from datasets import Dataset
from unsloth import FastLanguageModel, PatchDPOTrainer

def _dpo_unwrap_init(target):
    fn = target.__init__ if isinstance(target, type) else target
    seen = set()
    while fn is not None and id(fn) not in seen:
        seen.add(id(fn))
        w = getattr(fn, '__wrapped__', None)
        if w is not None:
            fn = w
            continue
        cl = getattr(fn, '__closure__', None)
        cd = getattr(fn, '__code__', None)
        if cl and cd and 'original_init' in getattr(cd, 'co_freevars', ()):
            fn = cl[cd.co_freevars.index('original_init')].cell_contents
            continue
        break
    return fn

def _dpo_safe_init(cls, raw):
    pairs = (('processing_class', 'tokenizer'), ('eval_strategy', 'evaluation_strategy'), ('max_length', 'max_seq_length'))
    def _ep(fn):
        try:
            sig = inspect.signature(fn)
            names = {p.name for p in sig.parameters.values() if p.name != 'self' and p.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY)}
            vkw = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())
            return names, vkw
        except (TypeError, ValueError):
            return set(), True
    explicit, leaf_vkw = _ep(_dpo_unwrap_init(cls))
    if isinstance(cls, type) and (leaf_vkw or not explicit):
        for b in cls.__mro__:
            if b is not object:
                explicit.update(_ep(_dpo_unwrap_init(b))[0])
    kw = dict(raw)
    for p1, p2 in pairs:
        if p1 in kw and p2 in kw:
            if explicit and p2 in explicit and p1 not in explicit:
                kw.pop(p1, None)
            else:
                kw.pop(p2, None)
    if explicit and (not leaf_vkw or isinstance(cls, type)):
        kw = {k: v for k, v in kw.items() if k in explicit}
    alt = {a: b for a, b in pairs} | {b: a for a, b in pairs}
    tried = set()
    for _ in range(8):
        try:
            return cls(**kw)
        except TypeError as exc:
            m = re.search(r"unexpected keyword argument '([^']+)'", str(exc))
            if not m or m.group(1) not in kw:
                raise
            bad = m.group(1)
            kw.pop(bad, None)
            fb = alt.get(bad)
            if fb and fb in raw and fb not in kw and fb not in tried:
                kw[fb] = raw[fb]
                tried.add(fb)
    return cls(**kw)

if RUN_DPO_STAGE and dpo_raw:
    PatchDPOTrainer()
    from trl import DPOConfig, DPOTrainer

    if hasattr(FastLanguageModel, 'for_training'):
        FastLanguageModel.for_training(model)
    else:
        model.train()

    def fmt_dpo(rec):
        msgs = []
        if rec.get('system'):
            msgs.append({'role': 'system', 'content': rec['system']})
        msgs.append({'role': 'user', 'content': rec['prompt']})
        return {
            'prompt': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True),
            'chosen': rec['chosen'] + '<|im_end|>\n',
            'rejected': rec['rejected'] + '<|im_end|>\n',
        }

    ds_dpo = Dataset.from_list([fmt_dpo(r) for r in dpo_raw])
    dpo_warmup_steps = max(3, int(0.08 * len(ds_dpo) / 16))
    max_p_len = min(2560, MAX_SEQ_LENGTH - 512)
    dpo_args = _dpo_safe_init(DPOConfig, {
        'output_dir': '/content/veraxus_out/dpo',
        'per_device_train_batch_size': 2,
        'gradient_accumulation_steps': 8,
        'warmup_steps': dpo_warmup_steps,
        'num_train_epochs': 1,
        'learning_rate': 1e-5,
        'beta': 0.1,
        'fp16': not USE_BF16,
        'bf16': USE_BF16,
        'logging_steps': 10,
        'save_strategy': 'no',
        'optim': 'adamw_8bit',
        'lr_scheduler_type': 'cosine',
        'seed': 2026,
        'report_to': 'none',
        'max_length': MAX_SEQ_LENGTH,
        'max_prompt_length': max_p_len,
    })
    dpo_trainer_kw = {
        'model': model,
        'ref_model': None,
        'args': dpo_args,
        'train_dataset': ds_dpo,
        'processing_class': tokenizer,
        'tokenizer': tokenizer,
    }
    for _k, _v in (('beta', 0.1), ('max_length', MAX_SEQ_LENGTH), ('max_prompt_length', max_p_len)):
        if not hasattr(dpo_args, _k):
            dpo_trainer_kw[_k] = _v
    dpo_trainer = _dpo_safe_init(DPOTrainer, dpo_trainer_kw)
    dpo_stats = dpo_trainer.train()
    print('DPO completed:', dpo_stats.metrics)
else:
    print('Skipped DPO stage.')


In [ ]:
# @title 7. Smoke Test JSON Output (SQLite + Python Sandbox + Unanswerable)
FastLanguageModel.for_inference(model)

smoke_prompts = [
    ('SQL', 'Bạn là chuyên gia viết SQLite cho DataMate. Trả DUY NHẤT JSON.', 'Bảng don_hang(ma_don, thanh_tien, tinh_trang). Tính tổng doanh thu hợp lệ?'),
    ('Python', 'Bạn là chuyên gia viết Python Sandbox (`query_df`, `sum_compound_vn_nums`) cho DataMate. Trả DUY NHẤT JSON.', 'Bảng so_chi(ngay, so_tien [320k + 150000]). Tính tổng tiền chi?'),
    ('Unanswerable', 'Bạn là chuyên gia viết SQLite cho DataMate. Trả DUY NHẤT JSON.', 'Hôm nay trời có mưa không?'),
]
for tag, sp, up in smoke_prompts:
    msgs = [{'role': 'system', 'content': sp}, {'role': 'user', 'content': up}]
    enc = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt')
    if hasattr(enc, 'keys') or isinstance(enc, dict):
        input_ids = enc['input_ids'].to('cuda')
        attn_mask = enc.get('attention_mask')
        gen_kw = {'input_ids': input_ids, 'max_new_tokens': 220, 'do_sample': False}
        if attn_mask is not None:
            gen_kw['attention_mask'] = attn_mask.to('cuda')
    else:
        input_ids = enc.to('cuda')
        gen_kw = {'input_ids': input_ids, 'max_new_tokens': 220, 'do_sample': False}
    out_ids = model.generate(**gen_kw)
    reply = tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
    print(f'[{tag}] -> {reply.strip()}')


In [ ]:
# @title 8. Xuất GGUF (Q4_K_M) + Sinh Ollama Modelfile + Tải về
import pathlib, re, shutil

EXPORT_DIR = pathlib.Path('/content/veraxus_export')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
GGUF_RAW_DIR = EXPORT_DIR / 'raw'
GGUF_RAW_DIR.mkdir(parents=True, exist_ok=True)
final_gguf = EXPORT_DIR / GGUF_FILENAME

def _find_exported_ggufs(roots, dst_path, quant='q4_k_m'):
    q_low = quant.lower()
    dst_res = dst_path.resolve()
    seen, out = set(), []
    for r in roots:
        if not r.exists():
            continue
        for p in r.rglob('*.gguf'):
            rp = p.resolve()
            if rp == dst_res or rp in seen:
                continue
            if q_low not in ('bf16', 'f16', 'f32') and re.search(r'\.(?:bf16|f16|f32)\.gguf$', p.name.lower()):
                continue
            seen.add(rp)
            out.append(p)
    out.sort(key=lambda p: (0 if q_low in p.name.lower() else 1, -p.stat().st_size))
    return out

_roots = [EXPORT_DIR, pathlib.Path(f'{GGUF_RAW_DIR}_gguf'), pathlib.Path('.')]
cands = _find_exported_ggufs(_roots, final_gguf, 'q4_k_m')
if not final_gguf.exists() and not cands:
    model.save_pretrained_gguf(str(GGUF_RAW_DIR), tokenizer, quantization_method='q4_k_m')
    cands = _find_exported_ggufs(_roots, final_gguf, 'q4_k_m')
else:
    print('Đã tìm thấy file GGUF đã convert sẵn, bỏ qua bước convert lại!')

if not final_gguf.exists():
    assert cands, 'Không tìm thấy file .gguf sau khi export!'
    shutil.move(str(cands[0]), str(final_gguf))

modelfile_txt = (
    f'FROM ./{GGUF_FILENAME}\n\n'
    'TEMPLATE """{{- if .Messages }}\n'
    '{{- range $i, $_ := .Messages }}\n'
    '<|im_start|>{{ .Role }}\n{{ .Content }}<|im_end|>\n'
    '{{- end }}\n<|im_start|>assistant\n{{- else }}\n'
    '{{- if .System }}<|im_start|>system\n{{ .System }}<|im_end|>\n{{- end }}\n'
    '{{- if .Prompt }}<|im_start|>user\n{{ .Prompt }}<|im_end|>\n{{- end }}\n'
    '<|im_start|>assistant\n{{- end }}{{ .Response }}{{- if .Response }}<|im_end|>{{- end }}"""\n\n'
    'PARAMETER stop "<|im_start|>"\n'
    'PARAMETER stop "<|im_end|>"\n'
    'PARAMETER stop "<|endoftext|>"\n'
    'PARAMETER temperature 0\n'
    'PARAMETER top_p 0.9\n'
    'PARAMETER num_ctx 4096\n'
)
(EXPORT_DIR / 'Modelfile').write_text(modelfile_txt, encoding='utf-8')
print(f'GGUF sẵn sàng: {final_gguf} ({final_gguf.stat().st_size / (1024**3):.2f} GB)')
print('Mẹo: nếu tải trình duyệt chậm, đặt SAVE_TO_GOOGLE_DRIVE = True rồi chạy lại Cell 8 (~5 giây, không convert lại).')

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drv = pathlib.Path(DRIVE_EXPORT_DIR)
    drv.mkdir(parents=True, exist_ok=True)
    shutil.copy2(final_gguf, drv / GGUF_FILENAME)
    shutil.copy2(EXPORT_DIR / 'Modelfile', drv / 'Modelfile')
    print(f'Đã lưu vào Google Drive: {drv}')
else:
    from google.colab import files
    files.download(str(EXPORT_DIR / 'Modelfile'))
    files.download(str(final_gguf))
